In [ ]:
import time
import datetime as dt
import pyvisa
from pathlib import Path

In [3]:
savepath = Path("C:/Users/IPMU/Desktop/ipmu_DAQ/Temperature")
savepath.mkdir(parents=True, exist_ok=True)

# look up instruments
rm = pyvisa.ResourceManager()
print("resource manager : ", rm)
rl = rm.list_resources()
print("List of resources: \n", rl)


resource manager :  Resource Manager of Visa Library at C:\WINDOWS\system32\visa32.dll
List of resources: 
 ('GPIB0::12::INSTR', 'GPIB1::12::INSTR')


In [4]:
for i,name in enumerate(rl):
    if "GPIB" in name:
        my_instrument = rm.open_resource(name)
        print("Resource[",i,"]:", name, "\n", my_instrument.query("*IDN?"))

Resource[ 0 ]: GPIB0::12::INSTR 
 LSCI,MODEL224,LSA17QI/OCD17QI/OCC17QI,1.1

Resource[ 1 ]: GPIB1::12::INSTR 
 LSCI,MODEL218S,21SAJP,042109



In [ ]:
LS224 = rm.open_resource(rl[0])
LS218 = rm.open_resource(rl[1])

LS224_CHANNELS = ["A", "B", "C1", "C2", "C3", "C4", "C5", "D1"]
LS218_CHANNELS = [f"Ch{i}" for i in range(1, 9)] # Ch1 - Ch8


In [6]:
filename = savepath / f"{dt.datetime.now().strftime('%Y%m%d%H%M%S')}_LS224_LS218.csv"
print(filename)

columnname = ['Abstime', 'Reltime', *LS224_CHANNELS, *LS218_CHANNELS]
try:
    with open(filename, mode="a") as f:
        print(*columnname, sep=", ", file=f)
except:
    pass

REFRESHRATE = 1  # [sec] for frequency of daq


C:\Users\IPMU\Desktop\ipmu_DAQ\Temperature\20260330214441_LS224_LS218.csv


In [7]:
def getTemp(self, ch=None):
    if ch == None:
        ch = "1"
    self.write("KRDG? %s" % ch)  # if ch=0, it mean readout all values
    T = float(self.read())
    return T

In [ ]:
StarTime = time.time()
while True:
    
    CurrentAbsTime = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    CurrentRerTime = time.time() - StarTime
    arr = [CurrentAbsTime, CurrentRerTime]

    #===LS224===
    for ch in LS224_CHANNELS:
        arr.append(getTemp(LS224, ch))

    #===LS218===
    for ch in range(1, 9):
        arr.append(getTemp(LS218, ch))

    print(arr)
    time.sleep(REFRESHRATE)

    try:
        
        with open(filename, mode="a") as f:
            print(*arr, sep=", ", file=f)

    except FileNotFoundError:

        print("FileNotFoundError... :'-(")
        filename = savepath / f"{dt.datetime.now().strftime('%Y%m%d%H%M%S')}_LS224_LS218.csv"
        with open(filename, mode="a") as f:
            print(*columnname, sep=", ", file=f)
            print(*arr, sep=", ", file=f)
